# 🦷 DentalVision AI — YOLOv8 Cavity Detection Training

This notebook trains a **YOLOv8** model on a custom dental cavity dataset.

**Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

---
| Item | Detail |
|------|--------|
| Model | YOLOv8n (nano) |
| Classes | 1 — `cavity` |
| Image Size | 640×640 |
| Epochs | 50 |
| Optimizer | AdamW |

## 1️⃣ Verify GPU & Install Dependencies

In [ ]:
# Verify GPU is enabled
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install Ultralytics YOLOv8
!pip install -q ultralytics
from ultralytics import YOLO
print("✓ Ultralytics installed successfully")

## 2️⃣ Upload & Extract Dataset

**Option A:** Upload `cavity_dataset.zip` from your local machine.  
**Option B:** Mount Google Drive if dataset is stored there.

Uncomment the option you prefer below.

In [ ]:
# ──────────────────────────────────────────────
# OPTION A: Upload zip directly from local machine
# ──────────────────────────────────────────────
from google.colab import files
print("Upload your cavity_dataset.zip file:")
uploaded = files.upload()

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')
print(f"✓ Extracted '{zip_name}' to /content/")

In [ ]:
# ──────────────────────────────────────────────
# OPTION B: Mount Google Drive (uncomment if needed)
# ──────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
#
# import zipfile
# zip_path = '/content/drive/MyDrive/cavity_dataset.zip'  # adjust path
# with zipfile.ZipFile(zip_path, 'r') as z:
#     z.extractall('/content/')
# print('✓ Dataset extracted from Google Drive')

In [ ]:
# Verify dataset structure
import os

base = '/content/cavity_dataset'
for split in ['train', 'val', 'test']:
    imgs = len(os.listdir(os.path.join(base, split, 'images')))
    lbls = len(os.listdir(os.path.join(base, split, 'labels')))
    print(f"  {split:6s} → {imgs} images, {lbls} labels")

# Peek at a label file
sample_label = os.listdir(os.path.join(base, 'train', 'labels'))[0]
print(f"\nSample label ({sample_label}):")
with open(os.path.join(base, 'train', 'labels', sample_label)) as f:
    print(f.read()[:200])

## 3️⃣ Create Dataset Configuration (dataset.yaml)

In [ ]:
# Auto-generate the YAML config required by Ultralytics
yaml_content = """path: /content/cavity_dataset
train: train/images
val: val/images
test: test/images

nc: 1
names:
  0: cavity
"""

yaml_path = '/content/cavity_dataset/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✓ Created {yaml_path}")
print(yaml_content)

## 4️⃣ Train YOLOv8 Model

Using `yolov8n.pt` pretrained weights with transfer learning.

| Hyperparameter | Value |
|----------------|-------|
| Epochs | 50 |
| Image Size | 640 |
| Batch Size | 16 |
| Optimizer | AdamW |
| Early Stopping | patience=10 |
| Augmentation | Mosaic, MixUp, Flip, HSV jitter |

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano model
model = YOLO('yolov8n.pt')

# Train on the dental cavity dataset
results = model.train(
    data='/content/cavity_dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    workers=2,
    project='runs/detect',
    name='train',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    verbose=True,
    seed=42,
)

print("\n✓ Training Complete!")
print(f"Best weights: runs/detect/train/weights/best.pt")

## 5️⃣ Evaluate on Validation Set

In [ ]:
# Load the best trained weights
best_model = YOLO('runs/detect/train/weights/best.pt')

# Validate on validation split
val_results = best_model.val(
    data='/content/cavity_dataset/dataset.yaml',
    split='val'
)

p = val_results.box.mp
r = val_results.box.mr
f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0

print("\n" + "="*50)
print("  VALIDATION METRICS")
print("="*50)
print(f"  mAP@50:     {val_results.box.map50:.4f}")
print(f"  mAP@50-95:  {val_results.box.map:.4f}")
print(f"  Precision:  {p:.4f}")
print(f"  Recall:     {r:.4f}")
print(f"  F1 Score:   {f1:.4f}")
print("="*50)

## 6️⃣ Evaluate on Test Set

In [ ]:
# Evaluate on the held-out test split
test_results = best_model.val(
    data='/content/cavity_dataset/dataset.yaml',
    split='test'
)

p_t = test_results.box.mp
r_t = test_results.box.mr
f1_t = 2 * (p_t * r_t) / (p_t + r_t) if (p_t + r_t) > 0 else 0

print("\n" + "="*50)
print("  TEST SET METRICS")
print("="*50)
print(f"  mAP@50:     {test_results.box.map50:.4f}")
print(f"  mAP@50-95:  {test_results.box.map:.4f}")
print(f"  Precision:  {p_t:.4f}")
print(f"  Recall:     {r_t:.4f}")
print(f"  F1 Score:   {f1_t:.4f}")
print("="*50)

## 7️⃣ Visualize Training Results

In [ ]:
from IPython.display import Image, display
import os

run_dir = 'runs/detect/train'

# Training curves (loss, mAP, precision, recall)
results_img = os.path.join(run_dir, 'results.png')
if os.path.exists(results_img):
    print("📊 Training Curves:")
    display(Image(filename=results_img, width=900))
else:
    print("results.png not found")

In [ ]:
# Confusion Matrix
cm_img = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(cm_img):
    print("🔢 Confusion Matrix:")
    display(Image(filename=cm_img, width=600))

cm_norm = os.path.join(run_dir, 'confusion_matrix_normalized.png')
if os.path.exists(cm_norm):
    print("\n🔢 Normalized Confusion Matrix:")
    display(Image(filename=cm_norm, width=600))

In [ ]:
# Sample Predictions on Validation Batch
for fname in sorted(os.listdir(run_dir)):
    if fname.startswith('val_batch') and fname.endswith('_pred.jpg'):
        print(f"\n🖼️ {fname}:")
        display(Image(filename=os.path.join(run_dir, fname), width=800))

## 8️⃣ Run Inference on Sample Test Images

In [ ]:
import glob

# Pick 4 random test images for visual check
test_images = sorted(glob.glob('/content/cavity_dataset/test/images/*'))[:4]

for img_path in test_images:
    results = best_model.predict(img_path, imgsz=640, conf=0.25, save=True,
                                  project='runs/detect', name='test_predictions', exist_ok=True)

# Display saved predictions
pred_dir = 'runs/detect/test_predictions'
if os.path.exists(pred_dir):
    for fname in sorted(os.listdir(pred_dir))[:4]:
        if fname.endswith(('.jpg', '.png')):
            print(f"\n🔍 {fname}:")
            display(Image(filename=os.path.join(pred_dir, fname), width=500))

## 9️⃣ Download Trained Model & Results

Download `best.pt` to use in your FastAPI backend.

In your backend, change:
```python
model = YOLO('yolov8n.pt')  # old
model = YOLO('best.pt')     # new
```

In [ ]:
from google.colab import files

# Download best.pt weights
best_pt = 'runs/detect/train/weights/best.pt'
if os.path.exists(best_pt):
    files.download(best_pt)
    print(f"✓ Downloading {best_pt}")
else:
    print("best.pt not found — training may not have completed.")

In [ ]:
# Download entire results folder as zip
import shutil

shutil.make_archive('training_results', 'zip', 'runs/detect/train')
files.download('training_results.zip')
print("✓ Downloading training_results.zip")

---
## ✅ Done!

**Next Steps:**
1. Place `best.pt` into `backend/` folder of your DentalVision AI project.
2. Update `main.py`: `model = YOLO('best.pt')`
3. Restart the FastAPI server.
4. Your cavity detection is now powered by your custom-trained model! 🎉